Завдання 1:
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [5]:
import os
import urllib.request
from datetime import datetime

def download_vhi_data(province_id):
    if not os.path.exists('data'):
        os.makedirs('data')
        
    existing_files = [f for f in os.listdir('data') if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують.")
        return

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    vhi_url = urllib.request.urlopen(url)
    
    date_time = datetime.now().strftime("%Y%m%d%H%M%S")
    filename = f"data/vhi_id_{province_id}_{date_time}.csv"
    
    with open(filename, 'wb') as f:
        f.write(vhi_url.read())
    print(f"VHI для області {province_id} завантажено.")

for i in range(1, 28):
    download_vhi_data(i)


Дані для області 1 вже існують.
Дані для області 2 вже існують.
Дані для області 3 вже існують.
Дані для області 4 вже існують.
Дані для області 5 вже існують.
Дані для області 6 вже існують.
Дані для області 7 вже існують.
Дані для області 8 вже існують.
Дані для області 9 вже існують.
Дані для області 10 вже існують.
Дані для області 11 вже існують.
Дані для області 12 вже існують.
Дані для області 13 вже існують.
Дані для області 14 вже існують.
Дані для області 15 вже існують.
Дані для області 16 вже існують.
Дані для області 17 вже існують.
Дані для області 18 вже існують.
Дані для області 19 вже існують.
Дані для області 20 вже існують.
Дані для області 21 вже існують.
Дані для області 22 вже існують.
Дані для області 23 вже існують.
Дані для області 24 вже існують.
Дані для області 25 вже існують.
Дані для області 26 вже існують.
Дані для області 27 вже існують.


Завдання 2 & 3:
2. Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області
3. Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [15]:
import os
import pandas as pd

def create_and_clean_dataframe(directory_path):

    provinces = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька",
        5: "Житомирська", 6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська",
        9: "Київська", 10: "Кіровоградська", 11: "Луганська", 12: "Львівська",
        13: "Миколаївська", 14: "Одеська", 15: "Полтавська", 16: "Рівненська",
        17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська",
        25: "Республіка Крим", 26: "Київ", 27: "Севастополь"
    }

    frames = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):

            province_id = int(filename.split('_')[2])

            df = pd.read_csv(os.path.join(directory_path, filename),
                             index_col=False,
                             header=1)

            df.columns = df.columns.str.strip().str.lower()

            df = df.rename(columns={col: 'vhi' for col in df.columns if 'vhi' in col})

            df['year'] = df['year'].astype(str).str.replace('<tt><pre>', '', regex=False)

            df = df[df['year'] != '']

            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            df['vhi'] = pd.to_numeric(df['vhi'], errors='coerce')

            df['province'] = province_id
            df['province_name'] = provinces.get(province_id, "Невідомо")

            frames.append(df.dropna(subset=['year', 'vhi']))

    return pd.concat(frames, ignore_index=True)


all_data = create_and_clean_dataframe('data')

all_data.head()

,year,week,smn,smt,vci,tci,vhi,province,province_name
0,1982.0,1.0,0.059,258.24,51.11,48.78,49.95,10,Кіровоградська
1,1982.0,2.0,0.063,261.53,55.89,38.20,47.04,10,Кіровоградська
2,1982.0,3.0,0.063,263.45,57.30,32.69,44.99,10,Кіровоградська
3,1982.0,4.0,0.061,265.10,53.96,28.62,41.29,10,Кіровоградська
4,1982.0,5.0,0.058,266.42,46.87,28.57,37.72,10,Кіровоградська


Завдання 4: Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [14]:
def print_vhi_extremes(df, province_id, start_year, end_year):
    subset = df[(df['province'] == province_id) & 
                (df['year'] >= start_year) & 
                (df['year'] <= end_year)]
    
    print(f"Аналіз VHI для області №{province_id} за період {start_year} - {end_year} рр:")
    
    if not subset.empty:
        print(f"Максимальний показник VHI: {subset['vhi'].max()}")
        print(f"Мінімальний показник VHI:  {subset['vhi'].min()}")
    else:
        print("Помилка: Дані за вказаний період відсутні.")

print_vhi_extremes(all_data, 10, 2000, 2020)

Аналіз VHI для області №10 за період 2000 - 2020 рр:
Максимальний показник VHI: 79.4
Мінімальний показник VHI:  -1.0


Завдання 5
Реалізація допоміжних функцій:
1. Отримання ряду VHI для вказаної області за конкретний рік.
2. Отримання даних VHI для кількох вибраних областей за вказаний діапазон років.

In [19]:
def get_vhi_for_year(df, province_id, year):
    print(f"Аналіз: Область №{province_id}, Рік: {year}")
    return df[(df['province'] == province_id) & (df['year'] == year)]

def get_vhi_range(df, province_ids, start_year, end_year):
    print(f"Аналіз: Області {province_ids}, Період: {start_year}-{end_year}")
    return df[(df['province'].isin(province_ids)) &
              (df['year'] >= start_year) &
              (df['year'] <= end_year)]

display(get_vhi_for_year(all_data, 7, 2010).head())

display(get_vhi_range(all_data, [7, 12], 2005, 2015).head())

Аналіз: Область №7, Рік: 2010


,year,week,smn,smt,vci,tci,vhi,province,province_name
55120,2010.0,1.0,0.068,255.21,25.53,73.41,49.47,7,Запорізька
55121,2010.0,2.0,0.067,254.40,25.60,75.65,50.63,7,Запорізька
55122,2010.0,3.0,0.065,253.90,24.21,78.23,51.22,7,Запорізька
55123,2010.0,4.0,0.060,253.65,20.31,81.77,51.04,7,Запорізька
55124,2010.0,5.0,0.062,254.68,19.46,82.03,50.74,7,Запорізька


Аналіз: Області [7, 12], Період: 2005-2015


,year,week,smn,smt,vci,tci,vhi,province,province_name
5668,2005.0,1.0,0.073,256.16,54.88,56.02,55.45,12,Львівська
5669,2005.0,2.0,0.075,255.87,52.04,57.73,54.89,12,Львівська
5670,2005.0,3.0,0.073,255.44,47.76,62.68,55.22,12,Львівська
5671,2005.0,4.0,0.072,255.05,45.66,68.01,56.84,12,Львівська
5672,2005.0,5.0,0.069,254.26,41.32,75.43,58.37,12,Львівська
